# Merge i agregacje w Pandas

Pandas = SQL w pamięci RAM.

| SQL | Pandas |
|---|---|
| `JOIN` | `df.merge()` |
| `GROUP BY` + `SUM/COUNT` | `df.groupby().agg()` |
| `LEFT JOIN` | `df.merge(..., how="left")` |

W tym notebooku pracujemy na trzech tabelach sklepu online:
`customers`, `orders`, `items`, ta sama struktura co Olist.

## Dane wejściowe

In [ ]:
import pandas as pd

customers = pd.DataFrame({
    "customer_id": [1, 2, 3, 4],
    "city":        ["Kraków", "Warszawa", "Gdańsk", "Poznań"],
    "segment":     ["vip", "regular", "regular", "vip"],
})

orders = pd.DataFrame({
    "order_id":    ["A", "B", "C", "D", "E", "F"],
    "customer_id": [  1,   1,   2,   3,   2,   4],
    "status":      ["delivered", "delivered", "delivered",
                    "cancelled", "delivered", "delivered"],
})

items = pd.DataFrame({
    "order_id": ["A", "A", "B", "C", "D", "E", "E", "F"],
    "product":  ["buty", "kurtka", "spodnie", "buty", "czapka",
                 "kurtka", "spodnie", "buty"],
    "price":    [ 120,    200,      80,       150,    40,
                  250,    90,       130],
    "qty":      [   1,      1,       2,         1,     1,
                    1,      1,        1],
})

print("customers:"); print(customers); print()
print("orders:");    print(orders);    print()
print("items:");     print(items)

## Merge 1: orders + customers

`LEFT JOIN`, każde zamówienie dostaje dane klienta.
Jeśli klient nie istnieje w `customers` → `NaN`.

In [ ]:
orders_with_city = orders.merge(customers, on="customer_id", how="left")

print(orders_with_city)

## Typy joinów: kiedy co używać

```python
how="left"   # wszystkie wiersze z lewej tabeli (najczęstszy w FE)
how="inner"  # tylko pasujące wiersze po obu stronach
how="outer"  # wszystkie wiersze z obu tabel
```

Sprawdź: co się stanie jeśli `customer_id=5` jest w `orders` ale nie ma go w `customers`?

In [ ]:
# Dodajemy zamówienie od nieznanego klienta
orders_extra = pd.concat([
    orders,
    pd.DataFrame({"order_id": ["G"], "customer_id": [99], "status": ["delivered"]}),
], ignore_index=True)

left_join  = orders_extra.merge(customers, on="customer_id", how="left")
inner_join = orders_extra.merge(customers, on="customer_id", how="inner")

print(f"LEFT JOIN:  {len(left_join)} wierszy , zamówienie G zostaje, city=NaN")
print(f"INNER JOIN: {len(inner_join)} wierszy , zamówienie G znika")
print()
print(left_join.tail(2))

## Problem: orders + items to relacja 1-do-wielu

Zamówienie A ma 2 produkty, po zwykłym merge dostaniemy duplikaty wierszy zamówienia.

In [ ]:
# ❌ Bezpośredni merge: powielone wiersze
naive_merge = orders.merge(items, on="order_id", how="left")
print("Wierszy po merge:", len(naive_merge), " (było", len(orders), "zamówień)")
print()
print(naive_merge.head(4))

## Rozwiązanie: najpierw zagreguj items, potem merguj

`groupby("order_id").agg(...)` → jedna linia na zamówienie.

In [ ]:
items_agg = (
    items
    .groupby("order_id")
    .agg(
        total_price  = ("price", "sum"),
        items_count  = ("product", "count"),
        avg_price    = ("price", "mean"),
        max_price    = ("price", "max"),
    )
    .reset_index()
)

print(items_agg)

In [ ]:
# ✅ Merge z zagregowaną tabelą: jeden wiersz na zamówienie
orders_full = orders.merge(items_agg, on="order_id", how="left")

print("Wierszy:", len(orders_full), ", tyle co zamówień")
print()
print(orders_full)

## Cechy behawioralne klienta

Chcemy wiedzieć o każdym kliencie: ile zamówień złożył, ile wydał łącznie.
To klasyczny przykład agregacji do Feature Engineering.

Uwaga: agregujemy tylko dostarczone zamówienia (`status == "delivered"`).

In [ ]:
customer_features = (
    orders_full[orders_full["status"] == "delivered"]
    .groupby("customer_id")
    .agg(
        orders_count  = ("order_id",    "count"),
        total_spent   = ("total_price", "sum"),
    )
    .assign(avg_order_value=lambda df: df["total_spent"] / df["orders_count"])
    .reset_index()
)

print(customer_features)

In [ ]:
# Dołączamy cechy klienta do tabeli klientów
customers_enriched = customers.merge(customer_features, on="customer_id", how="left")

# Klient 3 miał tylko anulowane zamówienie → NaN → wypełniamy 0
customers_enriched["orders_count"]  = customers_enriched["orders_count"].fillna(0)
customers_enriched["total_spent"]   = customers_enriched["total_spent"].fillna(0)
customers_enriched["avg_order_value"] = customers_enriched["avg_order_value"].fillna(0)

print(customers_enriched)

## Pełny łańcuch: od surowych tabel do gotowego DataFrame

To dokładnie ten sam wzorzec co w Olist:
`items → agg → merge z orders → merge z customers`

In [ ]:
result = (
    orders
    .merge(items_agg,                                    on="order_id",     how="left")
    .merge(customer_features,                            on="customer_id",  how="left")
    .merge(customers[["customer_id", "city", "segment"]], on="customer_id", how="left")
    .query("status == 'delivered'")
    .reset_index(drop=True)
)

print(result.to_string())

## W Olist zrobisz dokładnie to samo

| Ten notebook | Olist |
|---|---|
| `customers` | `olist_customers_dataset.csv` |
| `orders` | `olist_orders_dataset.csv` |
| `items` | `olist_order_items_dataset.csv` |
| `total_price` | suma `price + freight_value` |
| `customer_features` | historia zamówień klienta |